
# Three AGN recipes built by swapping selectors, not call sites

The composable AGN grammar (``agn.disc``, ``agn.torus``, ``agn.lines``,
``agn.feii``, ``agn.atten``) lets the user mix sub-blocks across model
families. Same SEDModel.build call, three different physics tuples:

1. **all-GRAHSP** — Buchner+2024 end-to-end (SBPL disc, GRAHSP lines,
   FeII forest, log-Gaussian torus, bi-attenuation).
2. **multicolor + SKIRTOR + NLR** — Kubota-Done disc with Stalevski
   clumpy torus and Gaussian NLR forest.
3. **QSOgen monolithic** — Temple+2021 empirical template as a single
   ``disc`` selector, no other blocks.

All three are evaluated at the same log L_bol = 12 in L_sun units
through ``SEDModel.build`` with the nested-dict grammar.

References: Buchner et al. 2024 (GRAHSP); Kubota & Done 2018;
Stalevski et al. 2016 (SKIRTOR); Temple, Hewett & Banerji 2021.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18
SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()

RECIPES = [
    (
        "all-GRAHSP",
        "tab:blue",
        {
            "disc": {"type": "grahsp_sbpl", "all_params": tengri.FIXED},
            "nlr": {"type": "grahsp", "all_params": tengri.FIXED},
            "blr": {"type": "grahsp", "all_params": tengri.FIXED},
            "feii": {"type": "grahsp", "all_params": tengri.FIXED},
            "torus": {"type": "grahsp", "all_params": tengri.FIXED},
            "atten": {"type": "grahsp_biatten", "all_params": tengri.FIXED},
        },
    ),
    (
        "multicolor disc + SKIRTOR torus + NLR",
        "tab:green",
        {
            "disc": {"type": "multicolor", "all_params": tengri.FIXED},
            "torus": {"type": "skirtor", "all_params": tengri.FIXED},
            "nlr": {"type": "analytic", "all_params": tengri.FIXED},
            "blr": {"type": "none", "all_params": tengri.FIXED},
        },
    ),
    (
        "QSOgen monolithic",
        "tab:orange",
        {"disc": {"type": "qsogen", "all_params": tengri.FIXED}},
    ),
]

fig, ax = plt.subplots(figsize=(8.0, 5.0))
for label, color, blocks in RECIPES:
    agn = {"all_params": tengri.FIXED, "log_lbol": 12.0, "lum_ratio": 1.0, **blocks}
    model = tengri.SEDModel.build(ssp, sfh=SFH, dust=DUST, agn=agn, redshift=tengri.Fixed(0.0))
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict(p)
    wave_um = np.asarray(model.wavelengths) * 1.0e-4
    nu_lnu = C_AA_PER_S / np.asarray(model.wavelengths) * np.asarray(out.rest_sed())
    ax.loglog(wave_um, np.where(nu_lnu > 0, nu_lnu, np.nan), lw=1.6, color=color, label=label)

ax.set(
    xlim=(5.0e-3, 1.0e2),
    ylim=(1.0e43, 1.0e47),
    xlabel=r"Rest-frame wavelength [$\mu$m]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
)
ax.legend(loc="lower center", fontsize=9, frameon=False)
fig.tight_layout()
plt.savefig("plot_composable_recipes.png", dpi=150, bbox_inches="tight")